In [2]:
# 1. 依存ライブラリのインストール
!pip install -q genesis-world
!apt-get install -y -qq ffmpeg

import numpy as np
import torch
import genesis as gs
from IPython.display import Video

# 2. 初期化（GPU優先、無ければCPUフォールバック）
use_gpu = torch.cuda.is_available()
backend = gs.gpu if use_gpu else gs.cpu

try:
    gs.init(backend=backend)
    print(f"Genesis initialized successfully on backend: {'GPU' if use_gpu else 'CPU'}")
except Exception as e:
    print(f"Reusing initialized Genesis session: {e}")

# 3. シーン作成
scene = gs.Scene(
    sim_options=gs.options.SimOptions(
        dt=0.01,
    ),
    show_viewer=False,
)

# 4. エンティティ配置
plane = scene.add_entity(gs.morphs.Plane())

# Franka Panda アーム
franka = scene.add_entity(
    gs.morphs.MJCF(file="xml/franka_emika_panda/panda.xml"),
)

# ブロックピラミッド
colors = [
    (0.9, 0.2, 0.2, 1.0),
    (0.2, 0.7, 0.2, 1.0),
    (0.2, 0.3, 0.9, 1.0),
    (0.9, 0.8, 0.1, 1.0),
]
block_size = (0.06, 0.06, 0.06)
tower_x = 0.65

blocks = []
c_idx = 0
for level in range(4):
    z = 0.03 + level * 0.065
    n_blocks = 4 - level
    y_start = -(n_blocks - 1) * 0.035
    for i in range(n_blocks):
        y = y_start + i * 0.07
        b = scene.add_entity(
            gs.morphs.Box(
                size=block_size,
                pos=(tower_x, y, z),
            ),
            surface=gs.surfaces.Rough(
                color=colors[c_idx % len(colors)],
            ),
        )
        blocks.append(b)
        c_idx += 1

# ストライク用のボール
ball = scene.add_entity(
    gs.morphs.Sphere(
        radius=0.045,
        pos=(0.35, 0.0, 0.045),
    ),
    surface=gs.surfaces.Rough(color=(0.95, 0.5, 0.1, 1.0)),
)

# レンダリング用カメラ
cam = scene.add_camera(
    res=(1280, 720),
    pos=(1.8, -1.3, 1.1),
    lookat=(0.4, 0.0, 0.3),
    fov=45,
    GUI=False,
)

# 5. ビルド
scene.build()

# 6. シミュレーション & 録画
cam.start_recording()

motors_dof = np.arange(7)

# ① 構え姿勢 (約1秒)
q_ready = np.array([0.0, -0.4, 0.0, -2.2, 0.0, 1.8, 0.78])
for _ in range(60):
    franka.control_dofs_position(q_ready, motors_dof)
    scene.step()
    cam.render()

# ② テイクバック (約0.75秒)
q_takeback = np.array([-0.3, -0.2, -0.2, -1.8, 0.0, 1.6, 0.78])
for _ in range(45):
    franka.control_dofs_position(q_takeback, motors_dof)
    scene.step()
    cam.render()

# ③ スイング＆強打 (約0.6秒)
q_strike = np.array([0.3, 0.6, 0.1, -0.8, 0.0, 2.4, 0.78])
for _ in range(40):
    franka.control_dofs_position(q_strike, motors_dof)
    scene.step()
    cam.render()

# ④ 崩壊の観察 (約3.3秒)
for _ in range(200):
    scene.step()
    cam.render()

# 7. 保存と表示
raw_video_path = "robot_strike.mp4"
cam.stop_recording(save_to_filename=raw_video_path, fps=60)
print(f"レンダリング完了: {raw_video_path}")

Video(raw_video_path, embed=True, width=720)

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
[Genesis] [22:01:48] [INFO] ╭───────────────────────────────────────────────╮
[Genesis] [22:01:48] [INFO] │┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈ Genesis ┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈│
[Genesis] [22:01:48] [INFO] ╰───────────────────────────────────────────────╯
[Genesis] [22:01:48] [INFO] Running on [Intel(R) Xeon(R) CPU @ 2.20GHz] with backend gs.cpu. Device memory: 12.67 GB.
[Genesis] [22:01:48] [INFO] 🚀 Genesis initialized. 🔖 version: 1.4.5, 🎨 theme: dark, 🌱 seed: None, 🐛 debug: False, 📏 precision: 32, 🔥 performance: False, 💬 verbose: INFO
Genesis initialized successfully on backend: CPU
[Genesis] [22:01:50] [INFO] Scene <7999e15> created.
[Genesis] [22:02:02] [INFO] Adding <gs.engine.entities.RigidEntity> plane_9aea998. idx: 0, morph: <gs.morphs.Plane>, material: <gs.materials.Rigid>.
[Genesis] [22:02:05] [WARNING] (MJCF) Approximating tendon by joint actuator for `finger_join

/usr/local/lib/python3.13/dist-packages/quadrants/lang/ast/ast_transformers/call_transformer.py:295: DeprecationWarning: keyword.__init__ got an unexpected keyword argument 'ctx'. Support for arbitrary keyword arguments is deprecated and will be removed in Python 3.15.
  kwarg_node = ast.keyword(


[Genesis] [22:04:36] [INFO] Building visualizer...
[Genesis] [22:04:43] [INFO] Software rendering context detected. Shadows and plane reflection not supported. Beware rendering will be extremely slow.
[Genesis] [22:04:46] [WARNING] fps=60 is not reachable with dt=0.01. Recording at fps=50.0 instead.
[Genesis] [22:05:21] [INFO] Running at 0.03 FPS.
[Genesis] [22:05:22] [INFO] Running at 0.09 FPS.
[Genesis] [22:05:22] [INFO] Running at 0.25 FPS.
[Genesis] [22:05:22] [INFO] Running at 0.32 FPS.
[Genesis] [22:05:23] [INFO] Running at 0.47 FPS.
[Genesis] [22:05:23] [INFO] Running at 0.53 FPS.
[Genesis] [22:05:24] [INFO] Running at 0.67 FPS.
[Genesis] [22:05:24] [INFO] Running at 0.72 FPS.
[Genesis] [22:05:25] [INFO] Running at 0.77 FPS.
[Genesis] [22:05:26] [INFO] Running at 0.79 FPS.
[Genesis] [22:05:26] [INFO] Running at 0.92 FPS.
[Genesis] [22:05:27] [INFO] Running at 0.96 FPS.
[Genesis] [22:05:28] [INFO] Running at 1.04 FPS.
[Genesis] [22:05:28] [INFO] Running at 1.03 FPS.
[Genesis] [22

TypeError: Camera.stop_recording() got an unexpected keyword argument 'save_to_filename'

In [6]:
import os
import glob
import shutil
from IPython.display import Video

# Genesisが自動生成したMP4ファイルを検索
mp4_files = glob.glob("*cam_0_*.mp4")

if mp4_files:
    latest_video = max(mp4_files, key=os.path.getctime)
    target_path = "/content/robot_strike.mp4"
    shutil.copy(latest_video, target_path)
    print(f"動画を検出・保存しました: {target_path}")

    # 画面上で再生
    display(Video(target_path, embed=True, width=720))
else:
    print("MP4ファイルが見つかりませんでした。")

動画を検出・保存しました: /content/robot_strike.mp4
